# Setup no Colab
Runtime: **Ambiente de execução → Alterar tipo → GPU**. Rode as células em ordem.
Dados e resultados ficam no Drive (`MyDrive/mestrado-epi`); o código vem do repositório.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

/bin/bash: line 1: nvidia-smi: command not found
3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


## 1. Drive (persistência de data/ e runs/)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/mestrado-epi'
os.makedirs(f'{DRIVE}/data', exist_ok=True); os.makedirs(f'{DRIVE}/runs', exist_ok=True)

## 2. Código
Preencha `REPO_URL` (repositório público no GitHub).

In [ ]:
REPO_URL = 'https://github.com/Guilh3rme/projeto-mestrado.git'
REPO = '/content/projeto-mestrado'
if os.path.exists(REPO): !git -C {REPO} pull
else: !git clone {REPO_URL} {REPO}
%cd {REPO}

## 3. Dependências
O Colab já traz PyTorch/CUDA; instala só o que falta. Usa `src/` direto (sem `pip install -e`).

In [ ]:
!pip install -q ultralytics pyyaml pytest
import sys
sys.path.insert(0, f'{REPO}/src')
# data/ e runs/ apontam para o Drive
!rm -rf data runs && ln -s {DRIVE}/data data && ln -s {DRIVE}/runs runs

## 4. Verificação

In [ ]:
import torch, ultralytics
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('ultralytics', ultralytics.__version__)
!python -m pytest -q

## 5. Dataset SH17 (Kaggle)
Credenciais: no Colab, painel de chave (Secrets) → crie `KAGGLE_USERNAME` e `KAGGLE_KEY` (kaggle.com → Settings → API) e ative o acesso ao notebook. Nunca escreva a chave no código.
O download vai para o Drive (uma vez só); o treino lê de uma cópia no disco local do Colab (mais rápido).

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
SH17 = f'{DRIVE}/data/sh17_raw'
if not os.path.exists(SH17):
    !pip install -q kaggle
    !kaggle datasets download -d mugheesahmad/sh17-dataset-for-ppe-detection -p {SH17} --unzip

In [ ]:
# Estrutura real do download (confira images/, labels/ e os *_files.txt antes de seguir)
for root, dirs, files in os.walk(SH17):
    if files: print(root.replace(SH17, '.'), len(files), files[:2])

## 6. Conversão para as 6 classes + split
Remapeia os rótulos (`ppe.data`), usa `train_files.txt`/`test_files.txt` oficiais e, se não houver `val_files.txt`, separa 10% do treino como validação (seed 0). O teste fica intocado até a Fase 6.

In [ ]:
import glob, random, shutil, yaml
from pathlib import Path
from ppe.data import CLASSES, convert_dir

def find(pattern):
    m = glob.glob(f'{SH17}/**/{pattern}', recursive=True)
    assert m, f'{pattern} não encontrado em {SH17}'
    return m[0]

IMG, LBL = Path(find('images')), Path(find('labels'))
OUT = Path('/content/sh17_ppe')
shutil.rmtree(OUT, ignore_errors=True)
convert_dir(LBL, OUT / 'labels_all')

def names(f): return [Path(l.strip()).name for l in open(find(f)) if l.strip()]
train, test = names('train_files.txt'), names('test_files.txt')
try: val = names('val_files.txt')
except AssertionError:
    random.Random(0).shuffle(train); k = len(train) // 10; val, train = train[:k], train[k:]

for split, lst in {'train': train, 'val': val, 'test': test}.items():
    (OUT / 'images' / split).mkdir(parents=True); (OUT / 'labels' / split).mkdir(parents=True)
    for img in lst:
        shutil.copy(IMG / img, OUT / 'images' / split / img)
        lab = OUT / 'labels_all' / (Path(img).stem + '.txt')
        (OUT / 'labels' / split / lab.name).write_text(lab.read_text() if lab.exists() else '')
    print(split, len(lst))

yaml.safe_dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
                'names': dict(enumerate(CLASSES))}, open('/content/ppe_colab.yaml', 'w'))

In [ ]:
# Sanidade: índices dos rótulos originais (esperado ≤ 16) e contagem por classe convertida
from collections import Counter
raw = Counter(int(l.split()[0]) for f in LBL.glob('*.txt') for l in f.read_text().splitlines() if l.strip())
assert max(raw) <= 16, raw
print('SH17 original:', sorted(raw.items()))
conv = Counter(int(l.split()[0]) for f in (OUT/'labels_all').glob('*.txt') for l in f.read_text().splitlines() if l.strip())
print({CLASSES[k]: v for k, v in sorted(conv.items())})

## 7. Treino (modelo)
Template; `project='runs'` grava no Drive. Ajuste quando o dataset/YAML existir.

In [ ]:
# from ultralytics import YOLO
# YOLO('yolov8s.pt').train(data='/content/ppe_colab.yaml', epochs=50, imgsz=640, batch=32,
#                           seed=0, project='runs', name='e1_baseline')